# Importations

In [52]:
import re
import csv
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

WRAP = 2 ** 32  # Période de débordement du compteur run-time (microsecondes)

# Affectation des tâches aux cœurs
CORE1 = {"audio", "ui", "IDLE1", "ipc1"}
CORE0 = {"maint", "esp_timer", "ipc0", "IDLE0",
         "BTC_TASK", "BTU_TASK", "btController", "hciT"}

# Régimes détectés automatiquement pour chaque fenêtre de 10 s.
# La zone filaire est subdivisée selon le format effectivement décodé, ce qui
# rend visible que le coût de la tâche audio suit l'effort de décodage.
REGIME_COLOR = {
    "repos":     "#eeeeee",  # aucune lecture
    "wav16":     "#dce9f5",  # WAV 44,1 kHz / 16 bits (le plus léger)
    "wav24":     "#debee4",  # WAV 44,1 kHz / 24 bits
    "mp3":       "#d9ead3",  # MP3 44,1 kHz (décodage Helix, le plus lourd)
    "bluetooth": "#f5e2dc",  # streaming A2DP
    "autre":     "#fbe5cd",  # tout autre format (repli)
}
REGIME_LABEL = {
    "repos":     "repos",
    "wav16":     "WAV 44,1 kHz / 16 bits",
    "wav24":     "WAV 44,1 kHz / 24 bits",
    "mp3":       "MP3 44,1 kHz",
    "bluetooth": "streaming Bluetooth",
}
# Ordre d'affichage dans la légende des régimes
REGIME_ORDRE = ["repos", "wav16", "wav24", "mp3", "bluetooth"]

# Charge Bluetooth = somme des tâches de la pile BT
BT_TASKS = ("BTC_TASK", "BTU_TASK", "btController", "hciT")
BT_SEUIL = 5.0     # % de charge cumulée au-delà duquel l'intervalle est classé streaming
AUDIO_SEUIL = 1.0  # % en deçà duquel on considère qu'aucune lecture n'a lieu

# Préparation des données

In [53]:
def strip_ansi(text):
    """Retire les séquences d'échappement ANSI et les retours chariot."""
    text = re.sub(r"\x1b\[[0-9;]*m", "", text)
    return text.replace("\r", "")

def parse_blocks(path):
    """Extrait la liste des blocs diag { t, tasks, rt, heap } du log."""
    # Note: Dans un notebook, assurez-vous que le fichier est dans le même dossier
    # ou fournissez le chemin absolu.
    try:
        content = open(path, encoding="utf-8", errors="replace").read()
    except FileNotFoundError:
        print(f"Erreur: Le fichier '{path}' est introuvable.")
        return []

    lines = strip_ansi(content).split("\n")
    blocks, cur = [], None

    for ln in lines:
        if re.match(r"^I \(\d+\) diag: ={4,} diagnostics ={4,}", ln):
            m = re.match(r"^I \((\d+)\)", ln)
            if m:
                cur = {"t": int(m.group(1)), "tasks": {}, "rt": {}, "heap": {}}
                blocks.append(cur)
            continue

        if cur is None:
            continue

        # Ligne de tâche
        m = re.match(
            r"^I \(\d+\) diag: (\S+)\s+(\S+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)%", ln)
        if m and m.group(1) not in ("task", "all", "run", "internal", "psram"):
            cur["tasks"][m.group(1)] = dict(
                core=m.group(2), prio=int(m.group(3)), stack=int(m.group(4)),
                hwm=int(m.group(5)), used=int(m.group(6)), pct=int(m.group(7)))
            continue

        # Ligne de run-time
        m = re.match(r"^(\S+)\s+\t(\d+)\t\t<?\d+%$", ln)
        if m:
            cur["rt"][m.group(1)] = int(m.group(2))
            continue

        # Heap interne
        m = re.match(r"^I \(\d+\) diag: internal\s+free\s+(\d+) B\s+min free ever\s+(\d+) B", ln)
        if m:
            cur["heap"]["int_free"] = int(m.group(1))
            cur["heap"]["int_min"] = int(m.group(2))
            continue

        # PSRAM
        m = re.match(r"^I \(\d+\) diag: psram\s+free\s+(\d+) B\s+min free ever\s+(\d+) B", ln)
        if m:
            cur["heap"]["ps_free"] = int(m.group(1))
            cur["heap"]["ps_min"] = int(m.group(2))

    return [b for b in blocks if b["tasks"]]


def parse_formats(path):
    """Timeline du format en cours de lecture, à partir des lignes pipeline.

    Le firmware imprime deux lignes par piste : le chemin du fichier (extension)
    puis la ligne "<rate> Hz <bits>-bit". On suit l'état courant et on émet un
    événement (t_s, ext, rate, bits) à chaque changement.
    """
    clean = strip_ansi(open(path, encoding="utf-8", errors="replace").read())
    events = []
    ext = rate = bits = None
    for ln in clean.split("\n"):
        m = re.match(r"^I \((\d+)\) pipeline: file: (/sdcard/.*)$", ln)
        if m:
            ext = m.group(2).rsplit(".", 1)[-1].lower()
            events.append((int(m.group(1)) / 1000.0, ext, rate, bits))
            continue
        m = re.match(r"^I \((\d+)\) pipeline: file: (\d+) Hz (\d+)-bit", ln)
        if m:
            rate, bits = int(m.group(2)), int(m.group(3))
            events.append((int(m.group(1)) / 1000.0, ext, rate, bits))
    return events


def format_lookup(events):
    """Retourne une fonction fmt_at(t) donnant (ext, rate, bits) à l'instant t."""
    def fmt_at(t):
        cur = (None, None, None)
        for te, ext, rate, bits in events:
            if te <= t:
                cur = (ext, rate, bits)
            else:
                break
        return cur
    return fmt_at


# Calcul charge CPU

In [54]:
def compute_load(blocks):
    """Charge par tâche (delta corrigé du débordement) pour chaque intervalle."""
    if not blocks:
        return [], []

    names = sorted({n for b in blocks for n in b["rt"]})
    rows = []

    for a, b in zip(blocks, blocks[1:]):
        dt = (b["t"] - a["t"]) * 1000.0  # ms -> us
        if dt <= 0:
            continue

        r = {"t_s": round(b["t"] / 1000.0, 1)}
        for n in names:
            if n in a["rt"] and n in b["rt"]:
                d = (b["rt"][n] - a["rt"][n]) % WRAP
            elif n in b["rt"]:
                d = b["rt"][n]
            else:
                d = 0
            r[n] = round(100.0 * d / dt, 2)

        r["charge_coeur1"] = round(sum(r.get(n, 0) for n in CORE1 if n != "IDLE1"), 2)
        r["charge_coeur0"] = round(sum(r.get(n, 0) for n in CORE0 if n != "IDLE0"), 2)
        rows.append(r)

    return names, rows


def format_regime(ext, rate, bits):
    """Associe un format de fichier à un régime coloré."""
    if ext == "mp3":
        return "mp3"
    if ext == "wav" and bits == 24:
        return "wav24"
    if ext == "wav":
        return "wav16"
    return "autre"


def annotate_formats(rows, fmt_at):
    """Attache à chaque intervalle le régime de format sous la clé _fmt."""
    for r in rows:
        r["_fmt"] = format_regime(*fmt_at(r["t_s"]))


def classify(r):
    """Classe un intervalle : streaming Bluetooth, format filaire, ou repos."""
    if sum(r.get(n, 0) for n in BT_TASKS) > BT_SEUIL:
        return "bluetooth"
    if r.get("audio", 0) < AUDIO_SEUIL:
        return "repos"
    return r.get("_fmt", "autre")


def segments(rows):
    """Regroupe les intervalles consécutifs de même régime en (début, fin, régime)."""
    segs = []
    for r in rows:
        reg = classify(r)
        t = r["t_s"] / 60.0
        if segs and segs[-1][2] == reg:
            segs[-1][1] = t
        else:
            segs.append([t, t, reg])
    return segs


# Création CSV

In [55]:
# --- CONFIGURATION ---
LOG_FILE = "../assets/data/campagne_0713_1356.log"  # <--- MODIFIEZ CECI

# Exécution du parsing et du calcul
blocks = parse_blocks(LOG_FILE)

if blocks:
    names, rows = compute_load(blocks)

    # Détection du format décodé pour chaque intervalle (colore la zone filaire)
    fmt_at = format_lookup(parse_formats(LOG_FILE))
    annotate_formats(rows, fmt_at)

    # --- BLOC DE SORTIE CSV ---
    cols = ["t_s"] + names + ["charge_coeur0", "charge_coeur1"]
    csv_path = "../assets/data/charge_cpu.csv"

    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)

    print(f"✅ Fichier créé : {csv_path}")
    print(f"   Données traitées : {len(rows)} intervalles")
else:
    print("❌ Aucun bloc diag trouvé. Vérifiez le nom du fichier log.")

✅ Fichier créé : ../assets/data/charge_cpu.csv
   Données traitées : 1258 intervalles


# Graphique charge CPU

In [ ]:
if blocks and rows:
    t = [r["t_s"] / 60 for r in rows]
    c0 = [r["charge_coeur0"] for r in rows]
    c1 = [r["charge_coeur1"] for r in rows]

    fig, ax = plt.subplots(figsize=(9, 3.6))

    # Fond coloré par régime détecté (repos / formats filaires / Bluetooth)
    for a, b, reg in segments(rows):
        ax.axvspan(a, b, color=REGIME_COLOR.get(reg, REGIME_COLOR["autre"]), zorder=0)

    line1, = ax.plot(t, c1, lw=0.7, color="#1f4e79", label="cœur 1 (audio, interface)")
    line0, = ax.plot(t, c0, lw=0.7, color="#a6371f", label="cœur 0 (Bluetooth, maintenance)")

    ax.set_xlim(0, t[-1])
    ax.set_ylim(0, 100)
    ax.set_xlabel("Temps (minutes)")
    ax.set_ylabel("Charge processeur (%)")

    # Légende des cœurs en haut à droite
    leg_lines = ax.legend(handles=[line1, line0], loc="upper right",
                          fontsize=8, framealpha=0.9)
    ax.add_artist(leg_lines)

    # Légende des régimes (fonds colorés) en haut à gauche
    patches = [Patch(facecolor=REGIME_COLOR[k], label=REGIME_LABEL[k]) for k in REGIME_ORDRE]
    ax.legend(handles=patches, loc="upper left", fontsize=8, framealpha=0.9)

    ax.grid(True, axis="y", lw=0.3, alpha=0.4)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

    fig.tight_layout()

    # --- BLOC DE SORTIE IMAGES ---
    pdf_path = "../assets/figures/charge_cpu.pdf"
    fig.savefig(pdf_path)

    print(f"✅ Fichiers créés : {pdf_path}")
    plt.show()
else:
    print("Aucune donnée à afficher.")

✅ Fichiers créés : ../assets/figures/charge_cpu.pdf


/tmp/ipykernel_3457/1319895550.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Graphique stackup

In [ ]:
if blocks and rows:
    pts = [(b["t"] / 1000 / 60, b["heap"]["int_free"] / 1024)
           for b in blocks if "int_free" in b["heap"]]

    if pts:
        t = [p[0] for p in pts]
        h = [p[1] for p in pts]

        fig, ax = plt.subplots(figsize=(9, 2.6))

        # Même fond de régimes que le graphe de charge, pour lecture croisée
        for a, b, reg in segments(rows):
            ax.axvspan(a, b, color=REGIME_COLOR.get(reg, REGIME_COLOR["autre"]), zorder=0)

        ax.plot(t, h, lw=0.7, color="#1f4e79")

        ax.set_xlim(0, t[-1])
        ax.set_ylim(0, max(h) * 1.1)
        ax.set_xlabel("Temps (minutes)")
        ax.set_ylabel("Tas interne libre (kio)")

        # Légende des régimes, en bas à gauche où la courbe laisse de la place
        patches = [Patch(facecolor=REGIME_COLOR[k], label=REGIME_LABEL[k]) for k in REGIME_ORDRE]
        ax.legend(handles=patches, loc="lower left", fontsize=7, framealpha=0.9, ncol=3)

        ax.grid(True, axis="y", lw=0.3, alpha=0.4)

        for s in ("top", "right"):
            ax.spines[s].set_visible(False)

        fig.tight_layout()

        # --- BLOC DE SORTIE IMAGES ---
        pdf_path = "../assets/figures/tas_interne.pdf"
        fig.savefig(pdf_path)

        print(f"✅ Fichiers créés : {pdf_path}")
        print(f"   Tas interne libre : min {min(h):.1f} kio, max {max(h):.1f} kio")
        plt.show()
    else:
        print("Aucune donnée de tas interne trouvée.")
else:
    print("Aucun bloc disponible.")

✅ Fichiers créés : ../assets/figures/tas_interne.pdf
   Tas interne libre : min 69.0 kio, max 140.7 kio


/tmp/ipykernel_3457/550495485.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Résumé

In [58]:
if blocks and rows:
    print("\n=== RÉCAPITULATIF ===")
    dur_h = (blocks[-1]["t"] - blocks[0]["t"]) / 3.6e6
    print(f"Durée : {dur_h:.2f} h, {len(blocks)} relevés")

    last = blocks[-1]["tasks"]
    print("\nPiles (HWM minimal sur la campagne) :")

    tasks_check = ["audio", "ui", "input", "maint"]
    for n in tasks_check:
        if n in last:
            hwms = [b["tasks"][n]["hwm"] for b in blocks if n in b["tasks"]]
            stk = last[n]["stack"]
            min_hwm = min(hwms) if hwms else 0
            occup_max = 100 * (stk - min_hwm) / stk if stk > 0 else 0
            print(f"  {n:6s} pile {stk:5d} o  marge min {min_hwm:5d} o  "
                  f"occup max {occup_max:.0f}%")
        else:
            print(f"  {n:6s} (non trouvée)")

    i0 = [r["charge_coeur0"] for r in rows]
    i1 = [r["charge_coeur1"] for r in rows]
    print(f"\nCharge crête cœur 0 : {max(i0):.1f}%   cœur 1 : {max(i1):.1f}%")
else:
    print("Impossible de générer le résumé (données manquantes).")


=== RÉCAPITULATIF ===
Durée : 3.57 h, 1259 relevés

Piles (HWM minimal sur la campagne) :
  audio  pile  4096 o  marge min  1544 o  occup max 62%
  ui     pile  8192 o  marge min  4348 o  occup max 47%
  input  pile  3072 o  marge min  2076 o  occup max 32%
  maint  pile  3072 o  marge min  1708 o  occup max 44%

Charge crête cœur 0 : 81.2%   cœur 1 : 63.7%
